# SkyQuery - Weather Agent

This notebook steps through each stage of the agent one at a time.
It runs in **demo mode** by default (synthetic data, no API keys). Set `DEMO = False`
and provide `GOOGLE_API_KEY` / `WEATHER_API_KEY` in your environment to use live services.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # make the package importable from /notebooks

from datetime import date
from weather_agent import Settings
from weather_agent.planner import plan_query
from weather_agent.pipeline import Pipeline
from weather_agent.formatter import to_text

DEMO = True

## 1. Pick the backends

In [ ]:
if DEMO:
    from weather_agent.demo import KeywordPlanner, SampleSource
    llm, source = KeywordPlanner(), SampleSource()
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from weather_agent.client import WeatherClient
    cfg = Settings.from_env(); cfg.require_live_keys()
    llm = ChatGoogleGenerativeAI(model=cfg.model_name, google_api_key=cfg.google_api_key, temperature=0)
    source = WeatherClient(cfg.weather_api_key)

## 2. Plan: free text → structured `QueryPlan`

In [ ]:
question = "Forecast for Paris and Tokyo for the next 3 days"
plan = plan_query(llm, question, today=date.today())
plan

## 3. Fetch: one call per city

In [ ]:
pipe = Pipeline(llm, source)
state = {"query": question, "plan": plan}
state.update(pipe.fetch(state))
len(state["reports"]), state["failures"]

## 4. Render

In [ ]:
print(to_text(state["reports"], state["failures"]))

## 5. Edge cases

In [ ]:
for q in ["who won the match?", "what's the weather like?", "past 2 days in London"]:
    out = pipe.run(q)
    print(f"{q!r:35} -> {out['status']}")